# Final Task B 17-class - Swin-Tiny

Train Swin-Tiny directly on the final 17 species classes. This uses ImageNet pretrained initialization and does not load the old 20-class Task B checkpoints.


In [2]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    ROOT = next((parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / 'src').exists()), Path.cwd())

sys.path.insert(0, str(ROOT))

import os
os.chdir(ROOT)
FINAL_B_DIR = ROOT / "notebooks" / "phase 1" / "task-B-17-new"
sys.path.insert(0, str(FINAL_B_DIR))
print(f"ROOT: {ROOT}")
print(f"task-B-17-new: {FINAL_B_DIR}")


ROOT: /workspace/woodbench
task-B-17-new: /workspace/woodbench/notebooks/phase 1/task-B-17-new


In [3]:
# Colab-only setup. Cloud data is expected to be already converted to 17 classes
# by 00_data_preparation.ipynb. This download only runs when the dataset is absent.
if Path('/content').exists():
    %pip install -q -r {ROOT}/requirements.txt

    DATA_DIR = ROOT / "data" / "task-B-group-split"
    if not DATA_DIR.exists():
        from huggingface_hub import login, snapshot_download
        import os

        token = os.getenv("HF_TOKEN", "")
        if token:
            login(token=token)
        snapshot_download(
            repo_id="ny0g507h3p/wood-task-B",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
            ignore_patterns=["*.md", "README*"],
        )


In [4]:
from final_b_17class import (
    BACKBONE_CONFIGS,
    prepare_environment,
    run_all_experiments,
    summarize_results,
    verify_17class_dataset,
)

prepare_environment(ROOT)

BACKBONE = "swin_tiny"
DISPLAY_NAME = "Swin-Tiny"
DATA_ROOT = ROOT / "data" / "task-B-group-split"
BATCH_SIZE = BACKBONE_CONFIGS[BACKBONE]["batch_size"]
EPOCHS = 50

print(f"Backbone: {DISPLAY_NAME} ({BACKBONE})")
print(f"Batch size: {BATCH_SIZE}")
print(f"Data root: {DATA_ROOT}")


Backbone: Swin-Tiny (swin_tiny)
Batch size: 32
Data root: /workspace/woodbench/data/task-B-group-split


In [5]:
class_names = verify_17class_dataset(DATA_ROOT)
print(f"Classes ({len(class_names)}):")
for idx, class_name in enumerate(class_names):
    print(f"  {idx:02d}: {class_name}")


[Data] train: 17 classes, 3365 images
[Data] val: 17 classes, 903 images
[Data] test: 17 classes, 532 images
Classes (17):
  00: Araucaria araucana
  01: Callitris columellaris
  02: Calocedrus rupestris
  03: Chamaecyparis obtusa
  04: Cryptomeria japonica
  05: Cunninghamia lanceolata
  06: Fokienia hodginsii
  07: Picea glauca
  08: Pinus cernua
  09: Pinus dalatensis
  10: Pinus krempfii
  11: Pinus radiata
  12: Podocarpus pilgeri
  13: Pseudotsuga brevifolia
  14: Taxus wallichiana
  15: Thujopsis dolabrata
  16: Xanthocyparis vietnamensis


In [6]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")

results = run_all_experiments(
    backbone_name=BACKBONE,
    root=ROOT,
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    device=DEVICE,
)


GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB
[Data] train: 17 classes, 3365 images
[Data] val: 17 classes, 903 images
[Data] test: 17 classes, 532 images
[Device] cuda
[Output] /workspace/woodbench/data/experiments/Task-B-new/final

main/seed_42/swin_tiny
[main | swin_tiny s42] train=3365, val=903, test=532, classes=17


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Epoch 1/50 | Train Loss: 2.8556, Acc: 6.58% | Val Loss: 2.8539, Acc: 5.54% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to swin_tiny_seed42_best.pth (Val Acc: 5.54%)


Epoch 2/50 | Train Loss: 1.7058, Acc: 62.83% | Val Loss: 0.8281, Acc: 93.69% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to swin_tiny_seed42_best.pth (Val Acc: 93.69%)


Epoch 3/50 | Train Loss: 0.7757, Acc: 95.57% | Val Loss: 0.6818, Acc: 97.67% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to swin_tiny_seed42_best.pth (Val Acc: 97.67%)


Epoch 4/50 | Train Loss: 0.6612, Acc: 98.69% | Val Loss: 0.6520, Acc: 98.56% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to swin_tiny_seed42_best.pth (Val Acc: 98.56%)


... (143 outputs left out)

Epoch 17/50 | Train Loss: 1.0131, Acc: 99.94% | Val Loss: 1.0243, Acc: 99.00% | LR: 8.35e-05[EarlyStop 7/10]


Epoch 18/50 | Train Loss: 1.0121, Acc: 99.88% | Val Loss: 1.0160, Acc: 99.78% | LR: 8.08e-05[EarlyStop 8/10]


Epoch 19/50 | Train Loss: 1.0154, Acc: 99.76% | Val Loss: 1.0227, Acc: 99.22% | LR: 7.80e-05[EarlyStop 9/10]


Epoch 20/50 | Train Loss: 1.0124, Acc: 99.88% | Val Loss: 1.0190, Acc: 99.56% | LR: 7.50e-05[EarlyStop 10/10]

[EarlyStopping] No improvement for 10 epochs. Stopping at epoch 20.
[taskB_label_smoothing_0.2 | swin_tiny s42] val_acc=99.89% test_acc=99.81% F1=99.83%


In [7]:
summarize_results(results)



Task B 17-class summary
experiment                       seed   val_acc  test_acc        f1      prec       rec
----------------------------------------------------------------------------------------
main                               42    99.56%    98.87%    98.97%    99.01%    99.02%
main                              123    99.89%    99.81%    99.83%    99.82%    99.84%
main                              456    99.89%    99.81%    99.82%    99.84%    99.81%
taskB_90rotation                   42    99.89%    98.87%    98.93%    99.03%    98.96%
taskB_label_smoothing_0.0          42   100.00%    99.62%    99.66%    99.65%    99.67%
taskB_label_smoothing_0.2          42    99.89%    99.81%    99.83%    99.82%    99.84%
----------------------------------------------------------------------------------------
main test_acc: 99.50 +/- 0.44%
main test_f1:  99.54 +/- 0.40%


## Continue after disk-full interruption

All six Swin-Tiny runs are complete. Run the setup cells above and the cells below only to reload and summarize the existing metrics; no training or resume is required.

In [8]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")


GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB


In [9]:
from final_b_17class import EXPERIMENTS, load_completed_results, train_one

continued_results = {
    (row["experiment"], row["seed"]): row
    for row in load_completed_results(backbone_name=BACKBONE, root=ROOT)
}

def run_remaining(experiment_name, seed):
    result = train_one(
        root=ROOT,
        backbone=BACKBONE,
        seed=seed,
        experiment_name=experiment_name,
        experiment_cfg=EXPERIMENTS[experiment_name],
        data_root=DATA_ROOT,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        device=DEVICE,
        resume_training=True,
    )
    continued_results[(experiment_name, seed)] = result
    return result

print(f"Loaded {len(continued_results)}/6 completed configs for {BACKBONE}.")


Loaded 6/6 completed configs for swin_tiny.


In [10]:
# Already complete: this only reuses the existing test_metrics.json.
run_remaining("taskB_label_smoothing_0.2", 42)


[taskB_label_smoothing_0.2 | swin_tiny s42] Reusing /workspace/woodbench/data/experiments/Task-B-new/final/taskB_label_smoothing_0.2/swin_tiny/test_metrics.json


{'backbone': 'swin_tiny',
 'seed': 42,
 'experiment': 'taskB_label_smoothing_0.2',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 99.88925802879291,
 'best_epoch': 10,
 'final_train_acc': 99.88095238095238,
 'final_val_acc': 99.55703211517165,
 'stopped_early': True,
 'epochs_trained': 20,
 'test_accuracy': 0.9981203007518797,
 'test_f1_macro': 0.9982665222101843,
 'test_precision_macro': 0.9982174688057041,
 'test_recall_macro': 0.9983660130718954,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 1.0,
  'Calocedrus rupestris': 1.0,
  'Chamaecypar

In [11]:
results = list(continued_results.values())
summarize_results(results)



Task B 17-class summary
experiment                       seed   val_acc  test_acc        f1      prec       rec
----------------------------------------------------------------------------------------
main                               42    99.56%    98.87%    98.97%    99.01%    99.02%
main                              123    99.89%    99.81%    99.83%    99.82%    99.84%
main                              456    99.89%    99.81%    99.82%    99.84%    99.81%
taskB_90rotation                   42    99.89%    98.87%    98.93%    99.03%    98.96%
taskB_label_smoothing_0.0          42   100.00%    99.62%    99.66%    99.65%    99.67%
taskB_label_smoothing_0.2          42    99.89%    99.81%    99.83%    99.82%    99.84%
----------------------------------------------------------------------------------------
main test_acc: 99.50 +/- 0.44%
main test_f1:  99.54 +/- 0.40%
